In [1]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Spin glass Hamiltonian ---
def spin_glass_energy(params, J):
    spins = torch.tanh(params)
    E = -torch.sum(J * torch.outer(spins, spins))
    return E / 2.0

# ----------------------------------------------------------------------
# NDDA computation (pure indicator, no update)
# ----------------------------------------------------------------------
def compute_NDDA(params, J, alpha=1.3):
    params = params.clone().detach().to(device).requires_grad_(True)
    d = params.numel()

    epsilon = torch.randn_like(params)
    params_1 = params + epsilon
    E1 = spin_glass_energy(params_1, J)
    E1.backward()
    I1 = epsilon * params.grad
    params.grad.zero_()

    mask_descent = (I1 < 0).to(device)
    mask_ascent  = (I1 > 0).to(device)
    epsilon_prime = torch.where(mask_descent, alpha * epsilon, epsilon)
    epsilon_prime = torch.where(mask_ascent, -epsilon, epsilon_prime)

    params_2 = params + epsilon_prime
    E2 = spin_glass_energy(params_2, J)
    E2.backward()
    I2 = epsilon_prime * params.grad
    params.grad.zero_()

    ndda_count = torch.sum((I1 >= 0) & (I2 >= 0)).float()
    return (ndda_count / d).item()

# ----------------------------------------------------------------------
# CPA optimizer with Hessian analysis
# ----------------------------------------------------------------------
def optimize_CPA(params_init, J, sigma_begin=0.02, sigma_end=0.001, epochs=200, alpha=1.3):
    J = J.clone().detach().to(device).requires_grad_(False)
    params = params_init.clone().detach().to(device).requires_grad_(True)

    losses = []
    ndda_vals = []
    min_eigenvalues = []       # most negative eigenvalue each 100 epochs
    len_of_params = params.numel()

    for epoch in range(epochs):
        sigma = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs


        # --- CPA two-probe ---
        epsilon = (torch.randn_like(params)).detach().to(device).requires_grad_(False)
        params_1 = params + epsilon
        E1 = spin_glass_energy(params_1, J)
        E1.backward()
        I1 = epsilon * params.grad
        params.grad.zero_()

        mask_descent = (I1 < 0).to(device)
        mask_ascent  = (I1 > 0).to(device)
        epsilon_prime = torch.where(mask_descent, alpha * epsilon, epsilon)
        epsilon_prime = (torch.where(mask_ascent, -epsilon, epsilon_prime)).detach().to(device).requires_grad_(False)

        params_2 = params + epsilon_prime
        E2 = spin_glass_energy(params_2, J)
        E2.backward()
        I2 = epsilon_prime * params.grad
        params.grad.zero_()

        # Full CPA update: I2 < 0 or (I1 < 0 and I2 >= 0)
        update_mask = (I2 < 0) | ((I1 < 0) & (I2 >= 0)).to(device)
        perturbation = torch.where((I2 < 0), epsilon_prime, epsilon)
        perturbation = perturbation * update_mask.float()

        with torch.no_grad():
            params += perturbation
            E_final = spin_glass_energy(params, J)
            losses.append(E_final.item() / N)

        ndda_vals.append(compute_NDDA(params, J, alpha))

        # ---- Hessian analysis every 100 epochs ----
        if epoch % 100 == 0:
            mu_Hessian = params.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]

            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad[i], mu_Hessian, create_graph=True)[0]
                H[i] = grad2
            eigenvalues, _ = torch.linalg.eig(H)
            eigenvalues = eigenvalues.real
            min_eigenvalues.append(torch.min(eigenvalues).item())

            # Store additional diagnostics (optional)
            # num_nonnegative = (eigenvalues >= 0).sum().item()
            # total = eigenvalues.numel()
            # etc.
        else:
            # keep last value for continuous time-series
            if len(min_eigenvalues) > 0:
                min_eigenvalues.append(min_eigenvalues[-1])
            else:
                min_eigenvalues.append(None)

        if epoch % 100 == 0:
            print(f"epoch: {epoch:5d}  CPA  loss/N: {losses[-1]:.6f}  NDDA: {ndda_vals[-1]:.4f} min_eigenvalue: {min_eigenvalues[-1]:.6f}")

    return losses, ndda_vals, min_eigenvalues

def switch_to_CPA(params_init, J, sigma, alpha=1.3):
    J = J.clone().detach().to(device).requires_grad_(False)
    params = params_init.clone().detach().to(device).requires_grad_(True)

    losses = []
    ndda_vals = []
    min_eigenvalues = []       # most negative eigenvalue each 100 epochs
    len_of_params = params.numel()


        # --- CPA two-probe ---
    epsilon = (torch.randn_like(params) * sigma).detach().to(device).requires_grad_(False)
    params_1 = params + epsilon
    E1 = spin_glass_energy(params_1, J)
    E1.backward()
    I1 = epsilon * params.grad
    params.grad.zero_()

    mask_descent = (I1 < 0).to(device)
    mask_ascent  = (I1 > 0).to(device)
    epsilon_prime = torch.where(mask_descent, alpha * epsilon, epsilon)
    epsilon_prime = (torch.where(mask_ascent, -epsilon, epsilon_prime)).detach().to(device).requires_grad_(False)

    params_2 = params + epsilon_prime
    E2 = spin_glass_energy(params_2, J)
    E2.backward()
    I2 = epsilon_prime * params.grad
    params.grad.zero_()

        # Full CPA update: I2 < 0 or (I1 < 0 and I2 >= 0)
    update_mask = (I2 < 0) | ((I1 < 0) & (I2 >= 0)).to(device)
    perturbation = torch.where((I2 < 0), epsilon_prime, epsilon)
    perturbation = perturbation * update_mask.float()
    with torch.no_grad():
        params += perturbation

    return params 

# ----------------------------------------------------------------------
# GD / PGD optimizer with Hessian analysis
# ----------------------------------------------------------------------
def optimize_GD_then_PGD_or_CPA(params_init, J, sigma_begin=0.02, sigma_end=0.001, escape_method = "CPA", epochs=10):
    method="GD"
    J = J.clone().detach().to(device).requires_grad_(False)
    params = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eigenvalues = []

    for epoch in range(epochs):
        lrlr = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs
        energy = spin_glass_energy(params, J)
        losses.append(energy.item() / N)
        
        if method == "GD":
            energy.backward()
            with torch.no_grad():
                   params -= lrlr * params.grad
        elif method == "PGD":
            energy.backward()
            with torch.no_grad():    
                noise = torch.randn_like(params)
                params -= lrlr * (params.grad + noise)
        elif method == "CPA":
            params = switch_to_CPA(params, J, sigma=lrlr)

        params.grad.zero_()
        ndda_vals.append(compute_NDDA(params, J))

        mu_Hessian = params.clone().detach().to(device).requires_grad_(True)
        y = spin_glass_energy(mu_Hessian, J)
        grad = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]
        max_grad_norm = torch.max(torch.abs(grad)).item()
        if max_grad_norm < 8e-3:
            temp = method
            method = escape_method
            if temp != escape_method:
                print(f"Switching to {escape_method} at epoch {epoch} due to small gradient norm: {max_grad_norm:.5e}")
        else:
            temp = method
            method="GD"
            if temp != "GD":
                print(f"Switching back to GD at epoch {epoch} due to large gradient norm: {max_grad_norm:.5e}")           

        # Hessian analysis every 100 epochs
        if epoch % 100 == 0:
            mu_Hessian = params.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]

            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad[i], mu_Hessian, create_graph=True)[0]
                H[i] = grad2
            eigenvalues, _ = torch.linalg.eig(H)
            eigenvalues = eigenvalues.real
            min_eigenvalues.append(torch.min(eigenvalues).item())
        else:
            if len(min_eigenvalues) > 0:
                min_eigenvalues.append(min_eigenvalues[-1])
            else:
                min_eigenvalues.append(None)

        if epoch % 100 == 0:
            print(f"epoch: {epoch:5d}  {method}  loss/N: {losses[-1]:.6f}  NDDA: {ndda_vals[-1]:.4f}  "
                  f"min_eigenvalue: {min_eigenvalues[-1]:.6f} max_grad_norm:{max_grad_norm:.5e}")

    return losses, ndda_vals, min_eigenvalues

# ----------------------------------------------------------------------
# Run experiment
# ----------------------------------------------------------------------
N = 1000
epoch_no = 40000
torch.manual_seed(0)

# J matrix construction (same as original)
J_1 = torch.randn((N//2, N//2), device=device) * 1
J_2 = torch.randn((N//2, N//2), device=device) * 3
J_3 = torch.randn((N//2, N//2), device=device) * 5
J = torch.rand((N, N), device=device)
J[:N//2, :N//2] = J_1 * math.sqrt(1/N)
J[N//2:, :N//2] = J_2 * math.sqrt(1/N)
J[:N//2, N//2:] = J_2 * math.sqrt(1/N)
J[N//2:, N//2:] = J_3 * math.sqrt(1/N)
J = (J + J.T) / 2
J = J * math.sqrt(1/N)
J.diagonal().zero_()

params0 = 0.4 * torch.randn(N, device=device)
logvar0 = -0.1 * torch.ones(N, device=device)   # not used in the new functions, but kept for compatibility

# Run optimizers
CPA_losses, CPA_ndda, CPA_min_eig = optimize_GD_then_PGD_or_CPA(params0, J, sigma_begin=0.4, sigma_end=0.001, escape_method="CPA", epochs=epoch_no)
PGD_losses, PGD_ndda, PGD_min_eig = optimize_GD_then_PGD_or_CPA(params0, J, sigma_begin=0.4, sigma_end=0.001, escape_method="PGD", epochs=epoch_no)

print("GD + CPA final loss:", CPA_losses[-1])
print("GD + PGD final loss:", PGD_losses[-1])


# ----------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------
# Energy
plt.figure(figsize=(9,5))
plt.plot(PGD_losses, label="GD + PGD")
plt.plot(CPA_losses, label="GD + CPA")
plt.xlabel("Epoch")
plt.ylabel("Energy")
plt.legend()
plt.title(f"Spin Glass Variational Optimization (N={N})")
plt.savefig("energy.png")
plt.close()

# NDDA index
plt.figure(figsize=(9,5))
plt.plot(PGD_ndda, label="GD + PGD")
plt.plot(CPA_ndda, label="GD + CPA")
plt.xlabel("Epoch")
plt.ylabel("NDDA")
plt.legend()
plt.title(f"NDDA Index (N={N})")
plt.savefig("ndda.png")
plt.close()

# Most negative eigenvalue (min eigenvalue)
plt.figure(figsize=(9,5))
# Only plot if values exist (skip Nones)
cpa_min_eig_arr = np.array([v if v is not None else np.nan for v in CPA_min_eig])
pgd_min_eig_arr = np.array([v if v is not None else np.nan for v in PGD_min_eig])
plt.plot(pgd_min_eig_arr, label="GD + PGD")
plt.plot(cpa_min_eig_arr, label="GD + CPA")
plt.xlabel("Epoch")
plt.ylabel("Minimum Eigenvalue")
plt.legend()
plt.title(f"Most Negative Eigenvalue (N={N})")
plt.savefig("min_eigenvalue.png")
plt.close()

epoch:     0  GD  loss/N: -0.000427  NDDA: 0.3360  min_eigenvalue: -0.143096 max_grad_norm:8.79663e-02
epoch:   100  GD  loss/N: -0.029413  NDDA: 0.0840  min_eigenvalue: -0.056209 max_grad_norm:7.90119e-02
epoch:   200  GD  loss/N: -0.038406  NDDA: 0.0300  min_eigenvalue: -0.038703 max_grad_norm:4.98939e-02
epoch:   300  GD  loss/N: -0.041365  NDDA: 0.0200  min_eigenvalue: -0.026141 max_grad_norm:2.39316e-02
epoch:   400  GD  loss/N: -0.042641  NDDA: 0.0350  min_eigenvalue: -0.025191 max_grad_norm:2.41324e-02
epoch:   500  GD  loss/N: -0.043354  NDDA: 0.0210  min_eigenvalue: -0.023836 max_grad_norm:3.23533e-02
epoch:   600  GD  loss/N: -0.043876  NDDA: 0.0220  min_eigenvalue: -0.020391 max_grad_norm:2.99366e-02
epoch:   700  GD  loss/N: -0.044312  NDDA: 0.0090  min_eigenvalue: -0.019979 max_grad_norm:3.96008e-02
epoch:   800  GD  loss/N: -0.044667  NDDA: 0.0110  min_eigenvalue: -0.010147 max_grad_norm:2.63779e-02
epoch:   900  GD  loss/N: -0.044891  NDDA: 0.0120  min_eigenvalue: -0.015

In [ ]:
cpa_losses, cpa_ndda, cpa_min_eig = optimize_CPA(params0, J, sigma_begin=0.2, sigma_end=0.001, epochs=epoch_no, alpha=1.3)
pgd_losses, pgd_ndda, pgd_min_eig = optimize_GD_or_PGD(params0, J, sigma_begin=0.2, sigma_end=0.001, method="PGD", epochs=epoch_no)

epoch:     0  PGD  loss/N: -0.000427  NDDA: 0.2040  min_eigenvalue: -0.153792
epoch:   100  PGD  loss/N: 0.002810  NDDA: 0.4320  min_eigenvalue: -0.129630
epoch:   200  PGD  loss/N: 0.000110  NDDA: 0.5640  min_eigenvalue: -0.145309
epoch:   300  PGD  loss/N: 0.001089  NDDA: 0.6370  min_eigenvalue: -0.071579
epoch:   400  PGD  loss/N: -0.002119  NDDA: 0.6670  min_eigenvalue: -0.104296
epoch:   500  PGD  loss/N: -0.002475  NDDA: 0.6900  min_eigenvalue: -0.077441
epoch:   600  PGD  loss/N: -0.000263  NDDA: 0.7050  min_eigenvalue: -0.101327
epoch:   700  PGD  loss/N: -0.001620  NDDA: 0.7410  min_eigenvalue: -0.196823
epoch:   800  PGD  loss/N: -0.000845  NDDA: 0.7390  min_eigenvalue: -0.169001
epoch:   900  PGD  loss/N: -0.000685  NDDA: 0.7650  min_eigenvalue: -0.061069
epoch:  1000  PGD  loss/N: -0.001296  NDDA: 0.7750  min_eigenvalue: -0.114413
epoch:  1100  PGD  loss/N: -0.000631  NDDA: 0.7890  min_eigenvalue: -0.084359
epoch:  1200  PGD  loss/N: -0.001680  NDDA: 0.8040  min_eigenvalue:

KeyboardInterrupt: 